In [2]:
import selenium
from selenium import webdriver

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [1]:
import time
import random

In [2]:
import pandas as pd
import numpy as np
import os
import re

In [3]:
import requests
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

In [4]:
from bs4 import BeautifulSoup

In [5]:
import aiohttp
import asyncio

In [5]:
data_stocknews= pd.read_csv("C:/Users/WQM/Desktop/本科毕业论文/数据集/FNSPID/Stocknews.csv", dtype= str)
data_stocknews= pd.DataFrame(data_stocknews)

In [6]:
data_stocknews['Year'] = pd.to_datetime(data_stocknews['Date']).dt.year

In [7]:
# 筛选年份在2010到2020之间的行
data_stocknews2 = data_stocknews[(data_stocknews['Year'] >= 2010) & (data_stocknews['Year'] <= 2020)]

In [8]:
# 整理dataframe顺序
data_stocknews2 = data_stocknews2.sort_values(by=['Date', 'Stock_symbol'], ascending=[True, True])
data_stocknews2.reset_index(drop= True, inplace= True)

In [9]:
data= data_stocknews2

In [12]:
data.to_csv("input.csv", index=False)

In [6]:
# request

In [7]:
data= pd.read_csv("C:/Users/WQM/Desktop/本科毕业论文/数据集/FNSPID/input.csv", dtype= str)
data= pd.DataFrame(data)

In [ ]:
# 定义一个函数来检查URL是否有效
def check_url(url):
    try:
        response = requests.get(url, timeout=5)
        # 如果响应码是200，则认为URL有效
        if response.status_code == 200:
            return True
        else:
            return False
    except requests.RequestException:
        # 如果请求失败，返回False
        return False

# 分批处理并保存结果的函数
def process_batch(batch_df, batch_num, output_file):
    with ThreadPoolExecutor(max_workers=50) as executor:
        futures = {executor.submit(check_url, url): url for url in batch_df['Url']}
        results = []
        for future in as_completed(futures):
            try:
                is_valid = future.result()
            except Exception:
                is_valid = False
            results.append(is_valid)
    
    # 将结果加入原 DataFrame
    batch_df.loc[:, 'is_url_valid'] = results
    
    # 保存有效的 URL
    valid_urls_df = batch_df.loc[batch_df['is_url_valid'] == True]
    valid_urls_df.drop(columns=['is_url_valid'], inplace=True)
    
    # 追加模式保存结果
    valid_urls_df.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
    
    print(f'Batch {batch_num+1} processed and saved.')

# 分批处理 DataFrame
def batch_process(df, batch_size, output_file):
    num_batches = len(df) // batch_size + 1
    
    for batch_num in range(start_batch, num_batches):
        start_idx = batch_num * batch_size
        end_idx = start_idx + batch_size
        batch_df = df.iloc[start_idx:end_idx]
        print(f'Processing batch {batch_num+1}/{num_batches}...')
        process_batch(batch_df, batch_num, output_file)
    
    print('All batches processed and saved.')

# 示例：加载 DataFrame 和调用批处理函数
start_batch= 305
batch_size = 10000  # 每批处理10000行
output_file = 'output.csv'  # 保存有效 URL 的文件

batch_process(data, batch_size, output_file)

In [ ]:
'''
input_file = 'C:/Users/WQM/Desktop/本科毕业论文/数据集/FNSPID/input.csv'

async def fetch_status(session, url):
    try:
        async with session.head(url, timeout=10) as response:
            return (url, response.status)
    except:
        return (url, None)

async def process_batch(batch_df):
    urls = batch_df['Url'].tolist()
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_status(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
    return results

def process_data(start_row=0, batch_size=1000):
    df = pd.read_csv('input.csv')  # 读取完整的DataFrame
    end_row = start_row + batch_size
    batch_df = df.iloc[start_row:end_row]  # 只处理需要的部分

    print(f"Processing rows {start_row} to {end_row}...")

    # 处理批次数据
    results = asyncio.run(process_batch(batch_df))

    # 提取有效的URL
    valid_urls = [url for url, status in results if status == 200]
    valid_df = batch_df[batch_df['Url'].isin(valid_urls)]
    
    # 保存有效数据到CSV
    output_file = f"output_{start_row}.csv"
    valid_df.to_csv(output_file, index=False)

    print(f"Processed and saved rows {start_row} to {end_row}.")

    return len(batch_df)

def process_in_batches(start_row=0, batch_size=1000):
    df = pd.read_csv('input.csv')  # 读取完整的DataFrame
    total_rows = len(df)
    
    while start_row < total_rows:
        num_processed = process_data(start_row, batch_size)
        if num_processed < batch_size:
            print(f"Completed processing all rows up to {start_row + num_processed}.")
            break
        start_row += batch_size

if __name__ == "__main__":
    start_row = 0  # 根据需要设置起始行
    process_in_batches(start_row)
'''

In [ ]:
# 使用apply函数来检查每个URL，标记那些不能访问的URL
data_sim['url_test'] = data_sim['Url'].apply(check_url)

# 删除那些URL无效的行
# data_sim = data_sim[data_sim['url_test'] == True].drop(columns=['url_test'])

In [ ]:
# 爬取数据
for index, row in data.head(5).iterrows():
    stock_symbol = row['Stock_symbol']  # 从DataFrame中获取股票代码
    url = row['Url']  # 从DataFrame中获取文章的URL
    
    response= urllib.request.urlopen(url)
    html= response.read().decode("utf-8")
    
    pattern = rf'[^.!?]*?\b{stock_symbol}\b[^.!?]*[.!?]'
    matches = re.findall(pattern, html)

    # 将匹配到的句子用句号连接
    text = '. '.join(matches)
    
    data.at[index, 'Article'] = text

In [41]:
# 使用BeautifulSoup解析HTML并提取可读文本
soup = BeautifulSoup(html, 'html.parser')
text = soup.get_text(separator=' ')  # 提取纯文本并将段落用空格分隔

In [47]:
# 去除多余的空白字符
text = re.sub(r'\s+', ' ', text).strip()

# 正则表达式提取包含指定股票代码的句子
stock_code = 'DRR'
pattern = rf'[^.!?]*?\b{stock_code}\b[^.!?]*[.!?]'
matches = re.findall(pattern, text)

# 去重：通过集合去重并保持顺序
unique_matches = list(dict.fromkeys(matches))

# 将匹配到的句子用句号连接
Article = '. '.join(unique_matches)

In [ ]:
# selenium 
# 配置Chrome选项
chrome_options = Options()
chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("-incognito") 

# 配置随机的 User-Agent 列表
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:89.0) Gecko/20100101 Firefox/89.0',
    'Mozilla/5.0 (Windows NT 10.0; WOW64; rv:45.0) Gecko/20100101 Firefox/45.0',
]

# 设置随机的 User-Agent
user_agent = random.choice(user_agents)
chrome_options.add_argument(f"user-agent={user_agent}")

# 启动浏览器
browser = webdriver.Chrome(options=chrome_options)

# 计数器，用于记录成功爬取的行数
success_count = 0

# 爬取数据
for index, row in data_stocknews2.head(5).iterrows():
    stock_code = row['Stock_symbol']  # 从DataFrame中获取股票代码
    url = row['Url']  # 从DataFrame中获取文章的URL
    
    try:
        # 打开目标网址
        browser.get(url)
        WebDriverWait(browser, 20).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'MsoNormal')]"))
        )

        # 随机等待几秒，模拟人为访问，防止反爬
        time.sleep(random.uniform(3, 7))

        # 使用更精确的XPath查找目标元素 (div class='MsoNormal')
        content_elements = WebDriverWait(browser, 10).until(
            EC.presence_of_all_elements_located(
                (By.XPATH, "//div[contains(@class, 'MsoNormal')]")
            )
        )
        
        # 合并所有元素的文本
        content = " ".join([element.text for element in content_elements])

        # 检查提取的文本中是否包含股票代码
        if stock_code in content:
            # 根据需求过滤出特定的句子
            sentences = content.split('. ')
            matched_sentences = [sentence.strip() for sentence in sentences if stock_code in sentence]
            
            if matched_sentences:
                # 直接将找到的句子替换到 'Article' 列中
                data_stocknews2.at[index, 'Article'] = ". ".join(matched_sentences) + ". "
                print(f"找到的句子: {data_stocknews2.at[index, 'Article']}")
            else:
                data_stocknews2.at[index, 'Article'] = ''
        else:
            data_stocknews2.at[index, 'Article'] = ''
        
        print(f"第 {index+1} 行：成功爬取到股票消息")
        success_count += 1
    except Exception as e:
        # 如果爬取失败，将该列置为空
        data_stocknews2.at[index, 'Article'] = ''
        print(f"第 {index+1} 行：未找到对应股票的消息，错误: {e}")
    
    # 随机等待几秒，避免频繁请求
    time.sleep(random.uniform(3, 7))

# 爬取完成后，打印成功爬取的行数
print(f"成功爬取了 {success_count} 行股票消息")

# 关闭浏览器
browser.quit()

In [17]:
pd.set_option('display.max_colwidth', None)  # 取消列内容的截断

In [42]:
# 防止被网站识别
chrome_options.add_experimental_option('excludeSwitches',['enable-automation'])
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("-incognito") 
browser = webdriver.Chrome(options=chrome_options)  # 参数名称变为options

In [47]:
url= "https://www.benzinga.com/news/20/05/16103463/71-biggest-movers-from-friday"
browser.get(url)
browser.implicitly_wait(10)

In [48]:
# 通过XPath查找包含 data-ticker="A" 的 <li> 元素
try:
    stock_code = 'A'
    stock_info = browser.find_element(By.XPATH, f"//span[@data-ticker='{stock_code}']/ancestor::li")
    print("找到的股票消息:", stock_info.text)
except Exception as e:
    print("未找到对应股票的消息:", e)

找到的股票消息: Agilent Technologies Inc A rose 5.2% to close at $84.98 after the company reported better-than-expected Q2 results.


In [36]:
stock_info.text

'SVB Leerink boosted the price target for Agilent Technologies Inc A from $90 to $95. Agilent closed at $80.75 on Thursday.'

In [37]:
stock_info

<selenium.webdriver.remote.webelement.WebElement (session="34e73480b55149403cc048119a9fb6f1", element="f.04441603732903F00E1E91DCF35A9FEA.d.9A22DBF947B0B3E863FC2B67BA979715.e.13")>

In [11]:
#browser = selenium.webdriver.Chrome() # 打开谷歌浏览器

In [38]:
browser.close() # 关闭当前标签页，如果只有一个则关闭浏览器

In [13]:
browser.quit()  # 彻底关闭，包括后台进程

In [ ]:
browser.get("url")

In [ ]:
# data-ticker= Stock_symbol

In [3]:
import urllib.request

In [4]:
response= urllib.request.urlopen("http://www.baidu.com")

In [6]:
html= response.read().decode("utf-8")